# lab Sheet 04 - Fine-Tuning an LLM using LoRA/QLoRA

Install the required libraries


In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes


In [ ]:
!pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Check GPU availability

In [ ]:

import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"GPU Memory: {gpu_memory:.2f} GB")
else:
    print("No GPU detected.")


GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


Select and load the LLM

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", model.device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


- The Qwen2.5-0.5B-Instruct model was successfully loaded using Hugging Face Transformers. The model runs on the Tesla T4 GPU, which provides sufficience memory for LoRA fine-tuning of this small language model.

Create a Dataset with 30 Question-Answer Pairs

In [ ]:

from datasets import Dataset

qa_pairs = [
    ("What products do you sell?", "We sell laptops, desktops, keyboards, mice, monitors, and computer accessories."),
    ("Do you sell gaming laptops?", "Yes, we offer gaming laptops suitable for different budgets."),
    ("Do you sell desktop computers?", "Yes, we sell desktop computers for office, home, and gaming use."),
    ("Can I buy a wireless mouse?", "Yes, wireless mice are available in our computer shop."),
    ("Do you sell mechanical keyboards?", "Yes, we offer mechanical keyboards for gaming and typing."),
    ("Do you have computer monitors?", "Yes, we sell monitors in different sizes and resolutions."),
    ("Do you sell laptop chargers?", "Yes, we sell compatible laptop chargers for various brands."),
    ("Can I upgrade my laptop RAM?", "Yes, we provide RAM upgrades for compatible laptops."),
    ("Do you sell SSD drives?", "Yes, we sell SSD drives for laptops and desktop computers."),
    ("Do you provide computer repairs?", "Yes, we provide laptop and desktop repair services."),
    ("Can you repair a broken laptop screen?", "Yes, we offer laptop screen replacement services."),
    ("Do you install Windows?", "Yes, we provide Windows installation services with a valid license."),
    ("Do you remove computer viruses?", "Yes, we provide malware scanning and removal services."),
    ("Can you recover deleted files?", "We may be able to recover deleted files depending on the storage condition."),
    ("Do you clean laptop cooling fans?", "Yes, we provide laptop cleaning and cooling system maintenance."),
    ("Do you sell USB flash drives?", "Yes, USB flash drives are available in different storage capacities."),
    ("Do you sell external hard drives?", "Yes, we sell external hard drives for data storage and backup."),
    ("Can I order a laptop online?", "Yes, customers can place laptop orders online."),
    ("Do you provide home delivery?", "Yes, home delivery is available for eligible locations."),
    ("How long does delivery take?", "Delivery time depends on your location and product availability."),
    ("Can I return a purchased product?", "Returns are accepted according to the product return policy."),
    ("Do your laptops have a warranty?", "Yes, laptops include warranty coverage according to the manufacturer."),
    ("Can I pay using a credit card?", "Yes, we accept credit card payments."),
    ("Do you offer installment payments?", "Installment payment options may be available for eligible purchases."),
    ("Can I get advice before buying a laptop?", "Yes, our staff can help you select a laptop based on your needs."),
    ("Which laptop is suitable for students?", "A laptop with at least 8GB RAM and an SSD is suitable for many student tasks."),
    ("Which computer is suitable for gaming?", "A gaming computer with a dedicated graphics card, sufficient RAM, and a modern processor is recommended."),
    ("Do you sell printers?", "Yes, we sell printers for home and office use."),
    ("Do you sell headphones?", "Yes, we offer wired and wireless headphones."),
    ("Can you build a custom PC?", "Yes, we can assemble custom desktop computers based on customer requirements.")
]

dataset = Dataset.from_list([
    {"question": q, "answer": a}
    for q, a in qa_pairs
])

print("Total Question-Answer Pairs:", len(dataset))
print("\nFirst Example:")
print(dataset[0])


Total Question-Answer Pairs: 30

First Example:
{'question': 'What products do you sell?', 'answer': 'We sell laptops, desktops, keyboards, mice, monitors, and computer accessories.'}


In [ ]:

def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful computer shop customer support assistant."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0]["text"])


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful computer shop customer support assistant.<|im_end|>
<|im_start|>user
What products do you sell?<|im_end|>
<|im_start|>assistant
We sell laptops, desktops, keyboards, mice, monitors, and computer accessories.<|im_end|>



Configure LoRA using PEFT

In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Fine-Tune the Model Using SFTTrainer

In [ ]:

from trl import SFTTrainer, SFTConfig

# Configure training
training_args = SFTConfig(
    output_dir="./computer_shop_lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    max_length=256,
    dataset_text_field="text",
    report_to="none"
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer
)

print("Trainer initialized successfully!")


Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Trainer initialized successfully!


In [ ]:
# Start fine-tuning
train_result = trainer.train()

print("Fine-tuning completed successfully!")
print("Training loss:", train_result.training_loss)


Step,Training Loss
1,5.105133
2,5.198438
3,5.287765
4,4.616343
5,4.224982
6,3.732085
7,3.351626
8,2.983154
9,2.787820
10,2.645290


Fine-tuning completed successfully!
Training loss: 3.063159245252609


In [ ]:
# Save LoRA adapters and tokenizer
trainer.save_model("./computer_shop_finetuned")
tokenizer.save_pretrained("./computer_shop_finetuned")

print("Fine-tuned LoRA adapters saved successfully!")


Fine-tuned LoRA adapters saved successfully!


Compare the Original and Fine-Tuned Models

In [ ]:
import torch

# Confirm the model and adapters are available
print("Model:", model_name)
print("Model device:", model.device)
print("LoRA adapters:", list(model.peft_config.keys()))

model.eval()
print("Models ready for comparison!")


Model: Qwen/Qwen2.5-0.5B-Instruct
Model device: cuda:0
LoRA adapters: ['default']
Models ready for comparison!


In [ ]:
test_questions = [
    "What products do you sell?",
    "Do you repair laptops?",
    "Can I buy a gaming laptop?",
    "Which laptop would you recommend for programming?",
    "Can you upgrade my laptop storage?",
    "What should I do if my laptop overheats?",
    "Do you provide delivery services?",
    "Can you help me build a gaming PC?"
]

print("Total test questions:", len(test_questions))


Total test questions: 8


In [ ]:
def generate_answer(question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful computer shop customer support assistant."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Apply chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract generated tokens only
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()



In [ ]:
question = "What products do you sell?"

answer = generate_answer(question)

print("Question:", question)
print("Answer:", answer)


Question: What products do you sell?
Answer: We offer a wide range of products, including but not limited to:

- Computers and peripherals
- Printers and copiers
- Network equipment
- Storage devices
- Servers and servers
- Networking equipment
- Security systems
- Home automation devices
- Smart home appliances
- Office supplies
- Electronics accessories


In [ ]:
results = []

for question in test_questions:

    # Original model
    with model.disable_adapter():
        original_answer = generate_answer(question)

    # Fine-tuned model
    finetuned_answer = generate_answer(question)

    results.append({
        "Question": question,
        "Original Answer": original_answer,
        "Fine-Tuned Answer": finetuned_answer
    })

    print("\nQuestion:", question)
    print("Original:", original_answer)
    print("Fine-Tuned:", finetuned_answer)
    print("-" * 60)



Question: What products do you sell?
Original: I'm sorry, but I don't have the capability to sell products as I am just a virtual assistant. However, I can help you with any questions or concerns you may have about my services!
Fine-Tuned: We offer a wide range of products, including but not limited to:

- Computers and peripherals
- Printers and copiers
- Network equipment
- Storage devices
- Servers and servers
- Networking equipment
- Security systems
- Home automation devices
- Smart home appliances
- Office supplies
- Electronics accessories
------------------------------------------------------------

Question: Do you repair laptops?
Original: Yes, I can help with repairing laptops! Please let me know if there's anything specific you need assistance with or if you have any questions about laptop repairs.
Fine-Tuned: Yes, we do repair laptop services.
------------------------------------------------------------

Question: Can I buy a gaming laptop?
Original: Yes, you can definite

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

display(results_df)


,Question,Original Answer,Fine-Tuned Answer
0,What products do you sell?,"I'm sorry, but I don't have the capability to ...","We offer a wide range of products, including b..."
1,Do you repair laptops?,"Yes, I can help with repairing laptops! Please...","Yes, we do repair laptop services."
2,Can I buy a gaming laptop?,"Yes, you can definitely buy a gaming laptop! G...","Yes, you can buy a gaming laptop."
3,Which laptop would you recommend for programming?,"For programming, I would recommend the MacBook...","For programming, the MacBook Air and the MacBo..."
4,Can you upgrade my laptop storage?,Certainly! I'd be happy to help you with that....,"Yes, we can upgrade your laptop's storage. Ple..."
5,What should I do if my laptop overheats?,"If your laptop overheats, it's important to ta...","If your laptop overheats, you can try the foll..."
6,Do you provide delivery services?,"Yes, we do offer delivery services to customer...","Yes, we do offer delivery services to customer..."
7,Can you help me build a gaming PC?,"Sure, I can help you with that! What kind of g...","Sure, I can help you with that! What kind of P..."


## Observations:
 During this practical activity I successfully fine-tuned the Qwen2.5-0.5B-Instruct language model using the LoRA technique. First, I installed the required libraries and checked GPU availability. The model was successfully loaded using a Tesla T4 GPU with 14.56 GB of memory.

I created a custom dataset containing 30 question-and-answer pairs related to computer shop customer support and formatted it using the model's chat template. Then, I configured LoRA using the PEFT library and fine-tuned the model for five training epochs using SFTTrainer. The training process was completed successfully with a training loss of approximately 3.06, and the fine-tuned LoRA adapters were saved.

Finally, I compared the original and fine-tuned models using eight test questions. The results showed that the fine-tuned model was able to provide more relevant responses to computer shop-related questions. However some generated answers were still too general or contained information beyond the training dataset. One of the main challenges was ensuring that the fine-tuned model generated accurate and relevant responses. This activity demonstrated how fine-tuning can adapt a pre-trained language model to a specific application.

## Learning Outcomes:
Through this practical activity, I gained practical knowledge of fine-tuning Large Language Models (LLMs) using the LoRA technique. I learned how to install and configure the required libraries, check GPU availability, and load a pre-trained language model using Hugging Face Transformers.

I also learned how to create and format a custom question-and-answer dataset, configure LoRA using the PEFT library, and fine-tune a model using SFTTrainer. Furthermore, I gained experience in monitoring training loss, saving fine-tuned model adapters, and comparing the responses of original and fine-tuned models.

Overall, this activity improved my understanding of parameter-efficient fine-tuning and how pre-trained language models can be adapted for specific tasks while reducing the number of trainable parameters and computational requirements.